# f6_m04c_sostenibilidad.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M04c — Sostenibilidad Computacional |

---

## 🎯 Qué hace

Mide la huella de carbono y el consumo computacional de los modelos principales.
Usa codecarbon para estimar emisiones de CO2 durante la inferencia.

Compara el modelo ganador (leído de `metricas_modelo.json`) con otros 2 modelos
representativos del top en términos de tiempo, memoria, CO2 y eficiencia
F1/CO2 (cum laude — métrica original de coste-rendimiento ambiental).

## 📋 Requisitos

- `data/06_evaluacion/metricas_modelo.json` — fuente única de verdad del ganador
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/<modelo_ganador>.pkl` (dinámico)
- `data/05_modelado/models/*.pkl` (otros modelos para comparativa)
- Paquete: codecarbon

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/sostenibilidad_metricas.parquet` | Tabla de métricas por modelo |
| `results/fase6/sostenibilidad_comparativa.png` | Comparativa tiempo/memoria/CO2 |
| `results/fase6/sostenibilidad_ratio_f1_co2.png` | 🏆 Eficiencia F1/CO2 |
| `docs/html/fase6/m04c_sostenibilidad.html` | Informe HTML |

## 🔄 Flujo

```
metricas_modelo.json → ganador dinámico
X_test_prep + 3 modelos (ganador + 2 representativos)
    ↓ Medir tiempo + memoria + CO2 (3 reps)
    ↓ Calcular F1 y ratio F1/CO2 (eficiencia)
    ↓ Gráfico comparativo + ratio
    → sostenibilidad_metricas.parquet + m04c_sostenibilidad.html
```

## ➡️ Siguiente

`f6_m06_informe_final.ipynb` — síntesis de todos los resultados


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

# JSON del ganador dinámico
RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'

print(f'ROOT:        {ROOT}')
print(f'DIR_MODELS:  {DIR_MODELS}')
print(f'DIR_RESULTS: {DIR_RESULTS}')
print(f'RUTA_JSON:   {RUTA_JSON}')

ROOT:        c:\PRUEBAS\AU_UJI_v2_RUTA_B
DIR_MODELS:  c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\05_modelado\models
DIR_RESULTS: c:\PRUEBAS\AU_UJI_v2_RUTA_B\results\fase6
RUTA_JSON:   c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\06_evaluacion\metricas_modelo.json


In [2]:
# ============================================================
# CELDA 2: IMPORTS Y CARGA DEL GANADOR DINÁMICO
# Sistema dinámico: el ganador se lee de metricas_modelo.json
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import time
import tracemalloc
from codecarbon import EmissionsTracker
from sklearn.metrics import f1_score
from src.html.render import render_pagina
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

plt.rcParams['figure.dpi'] = 120

# Cargar metadatos del modelo ganador (sistema dinámico)
assert RUTA_JSON.exists(), f'❌ No encontrado: {RUTA_JSON}'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
nombre_ganador     = meta_json['modelo_nombre']
familia_ganador    = meta_json['modelo_familia']

print('Imports OK.')
print(f'Modelo ganador: {nombre_ganador} ({familia_ganador})')
print(f'PKL:            {nombre_ganador_pkl}')

Imports OK.
Modelo ganador: LightGBM (Gradient Boosting)
PKL:            LightGBM__none.pkl


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y 3 MODELOS REPRESENTATIVOS
# Comparativa coste-rendimiento entre 3 puntos del espectro:
#   - GANADOR (LightGBM):    eficiente, gradient boosting, alto F1
#   - Stacking (ensemble):   complejo (23 MB), esperamos alta huella CO2
#   - EBM (interpretable):   simple, explicable, esperamos baja huella CO2
# Esta selección permite contar la historia coste-rendimiento ambiental.
# ============================================================
X_test_prep = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test      = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
y_true      = y_test.values.ravel()

# El ganador se carga dinámicamente desde el JSON.
# Stacking y EBM se buscan en DIR_MODELS — si alguno no existe se omite.
modelos_candidatos = {
    nombre_ganador: nombre_ganador_pkl,
    'Stacking':     'Stacking__balanced.pkl',
    'EBM':          'EBM__none.pkl',
}

modelos = {}
for nombre, pkl in modelos_candidatos.items():
    ruta = DIR_MODELS / pkl
    if ruta.exists():
        modelos[nombre] = joblib.load(ruta)
        print(f'  ✅ {nombre:15s} cargado ({pkl})')
    else:
        print(f'  ⚠️  {nombre:15s} no encontrado: {pkl}')

assert len(modelos) >= 2, '❌ Se necesitan al menos 2 modelos para comparativa.'

print(f'\nX_test_prep: {X_test_prep.shape}')
print(f'Modelos cargados: {list(modelos.keys())}')

  ✅ LightGBM        cargado (LightGBM__none.pkl)

  ✅ Stacking        cargado (Stacking__balanced.pkl)
  ✅ EBM             cargado (EBM__none.pkl)

X_test_prep: (6725, 27)
Modelos cargados: ['LightGBM', 'Stacking', 'EBM']


In [4]:
# ============================================================
# CELDA 4: MEDIR TIEMPO, MEMORIA Y CO2 POR MODELO
# Para cada modelo medimos:
#   - Tiempo de inferencia sobre X_test_prep (ms por observación)
#   - Memoria pico durante la inferencia (MB)
#   - Emisiones CO2 estimadas (gramos) usando codecarbon
# Repetimos 3 veces y tomamos la media para reducir variabilidad.
# ============================================================
N_REPS = 3
resultados = []

for nombre, modelo in modelos.items():
    print(f'Midiendo {nombre}...')
    tiempos  = []
    memorias = []

    for rep in range(N_REPS):
        # Tiempo
        t0 = time.perf_counter()
        _ = modelo.predict_proba(X_test_prep)
        t1 = time.perf_counter()
        tiempos.append(t1 - t0)

        # Memoria
        tracemalloc.start()
        _ = modelo.predict_proba(X_test_prep)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memorias.append(pico / 1024 / 1024)  # bytes → MB

    # CO2 — codecarbon necesita unos segundos mínimos para señal medible
    tracker = EmissionsTracker(
        output_dir=str(DIR_RESULTS), log_level='error', save_to_file=False
    )
    tracker.start()
    for _ in range(10):  # repetir para tener señal medible
        _ = modelo.predict_proba(X_test_prep)
    emisiones = tracker.stop()  # kg CO2
    emisiones_g = (emisiones or 0) * 1000 / 10  # gramos por inferencia

    tiempo_medio  = np.mean(tiempos)
    memoria_media = np.mean(memorias)
    ms_por_obs    = tiempo_medio / len(X_test_prep) * 1000

    y_prob_m = modelo.predict_proba(X_test_prep)[:, 1]
    f1_m     = f1_score(y_true, (y_prob_m >= 0.5).astype(int))

    resultados.append({
        'modelo':      nombre,
        'tiempo_s':    round(tiempo_medio, 3),
        'ms_por_obs':  round(ms_por_obs, 4),
        'memoria_mb':  round(memoria_media, 2),
        'co2_g':       round(emisiones_g, 6),
        'f1':          round(f1_m, 4),
        'f1_por_co2':  round(f1_m / max(emisiones_g, 1e-9), 2),
        'es_ganador':  nombre == nombre_ganador,
    })
    print(f'  Tiempo: {tiempo_medio:.3f}s | {ms_por_obs:.4f}ms/obs | '
          f'Memoria: {memoria_media:.1f}MB | CO2: {emisiones_g:.6f}g | F1: {f1_m:.4f}')

df_sos = pd.DataFrame(resultados)
df_sos.to_parquet(DIR_RESULTS / 'sostenibilidad_metricas.parquet')
print('\n✅ Métricas guardadas.')
print(df_sos.to_string(index=False))

Midiendo LightGBM...


[codecarbon WARNING @ 14:04:42] Multiple instances of codecarbon are allowed to run at the same time.


  Tiempo: 0.079s | 0.0117ms/obs | Memoria: 0.2MB | CO2: 0.000237g | F1: 0.8334
Midiendo Stacking...


  Tiempo: 0.227s | 0.0337ms/obs | Memoria: 3.7MB | CO2: 0.000365g | F1: 0.8273
Midiendo EBM...


  Tiempo: 0.003s | 0.0005ms/obs | Memoria: 1.8MB | CO2: 0.000143g | F1: 0.8071

✅ Métricas guardadas.
  modelo  tiempo_s  ms_por_obs  memoria_mb    co2_g     f1  f1_por_co2  es_ganador
LightGBM     0.079      0.0117        0.21 0.000237 0.8334     3513.56        True
Stacking     0.227      0.0337        3.67 0.000365 0.8273     2264.03       False
     EBM     0.003      0.0005        1.76 0.000143 0.8071     5654.37       False


In [5]:
# ============================================================
# CELDA 5: GRÁFICOS COMPARATIVOS
# 3 subplots: tiempo de inferencia, memoria pico, emisiones CO2.
# Gráfico ratio F1/CO2 separado (cum laude).
# Paleta UJI 2026:
#   azul  (#1e4d8c) = ganador (LightGBM)
#   rojo  (#dc2626) = Stacking (modelo costoso)
#   verde (#10b981) = EBM (modelo eficiente)
# ============================================================
PALETA_MODELOS = {
    nombre_ganador: '#1e4d8c',  # COLORES["primario"] — ganador
    'Stacking':     '#dc2626',  # COLORES["abandono"] — costoso
    'EBM':          '#10b981',  # COLORES["exito"]    — eficiente
}
colores = [PALETA_MODELOS.get(m, '#94a3b8') for m in df_sos['modelo']]

# --- Comparativa absoluta: tiempo, memoria, CO2 ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metricas = [
    ('ms_por_obs', 'Tiempo (ms/obs)',      'Tiempo de inferencia por observación'),
    ('memoria_mb', 'Memoria pico (MB)',    'Consumo de memoria durante inferencia'),
    ('co2_g',      'CO2 (g/inferencia)',   'Emisiones estimadas por inferencia completa'),
]

for ax, (col, ylabel, titulo) in zip(axes, metricas):
    bars = ax.bar(df_sos['modelo'], df_sos[col], color=colores, alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(titulo, fontsize=10)
    ax.tick_params(axis='x', rotation=15, labelsize=9)
    # Anotar valores encima de cada barra
    for bar, val in zip(bars, df_sos[col]):
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                    f'{val:.4f}', ha='center', fontsize=8)

plt.suptitle(
    f'Fase 6 — Sostenibilidad computacional: Comparativa de modelos (ganador: {nombre_ganador})',
    fontsize=12, y=1.02
)
plt.tight_layout()
ruta_sos = DIR_RESULTS / 'sostenibilidad_comparativa.png'
plt.savefig(ruta_sos, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Comparativa guardada: {ruta_sos.name}')

# --- Ratio F1/CO2 (cum laude) ---
ruta_ratio = None
if df_sos['co2_g'].sum() > 0:
    fig2, ax2 = plt.subplots(figsize=(8, 4.5))
    bars2 = ax2.bar(df_sos['modelo'], df_sos['f1_por_co2'],
                    color=colores, alpha=0.85)
    for bar, val in zip(bars2, df_sos['f1_por_co2']):
        if bar.get_height() > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                     f'{val:.1f}', ha='center', fontsize=9)
    ax2.set_ylabel('F1 / gramo CO2', fontsize=11)
    ax2.set_title(
        f'Fase 6 — Eficiencia ambiental: Rendimiento por coste\n'
        f'(F1 por gramo de CO2 emitido — ganador: {nombre_ganador})',
        fontsize=11
    )
    ax2.tick_params(axis='x', rotation=15)
    plt.tight_layout()
    ruta_ratio = DIR_RESULTS / 'sostenibilidad_ratio_f1_co2.png'
    plt.savefig(ruta_ratio, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ Ratio F1/CO2 guardado: {ruta_ratio.name}')
else:
    print('⚠️  CO2 = 0 (codecarbon sin señal suficiente) — ratio no calculado.')

✅ Comparativa guardada: sostenibilidad_comparativa.png


✅ Ratio F1/CO2 guardado: sostenibilidad_ratio_f1_co2.png


In [6]:
# ============================================================
# CELDA 6: GENERAR HTML
# render_pagina — estándar del proyecto.
# Incluye bloque Wilcoxon dinámico para justificar el modelo ganador.
# Sostenibilidad destacada: bloque verde con conclusión coste-rendimiento.
# Paleta UJI 2026 (alineada con config_app.py).
# ============================================================
import base64
from src.html.wilcoxon_block import bloque_wilcoxon_html

COLOR_PRIMARIO = '#1e4d8c'
COLOR_ABANDONO = '#dc2626'
COLOR_EXITO    = '#10b981'

def img_b64(ruta) -> str:
    if not ruta or not Path(ruta).exists():
        return ''
    with open(ruta, 'rb') as fh:
        return base64.b64encode(fh.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:{COLOR_ABANDONO}">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

# Tabla con resaltado de ganador y mejor eficiencia
mejor_eficiencia = df_sos.loc[df_sos['f1_por_co2'].idxmax(), 'modelo']
peor_co2         = df_sos.loc[df_sos['co2_g'].idxmax(), 'modelo']

filas_tabla = ''
for _, row in df_sos.iterrows():
    bg = '#f0fff4' if row['modelo'] == mejor_eficiencia else ('#f7fafc' if row['es_ganador'] else '')
    icono_ganador = ' 🏆' if row['es_ganador'] else ''
    icono_eco     = ' 🌱' if row['modelo'] == mejor_eficiencia else ''
    filas_tabla += (
        f'<tr style="background:{bg}">'
        f'<td style="padding:8px 12px;font-weight:600">{row["modelo"]}{icono_ganador}{icono_eco}</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["tiempo_s"]:.3f}s</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["ms_por_obs"]:.4f}</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["memoria_mb"]:.1f}</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["co2_g"]:.6f}</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["f1"]:.4f}</td>'
        f'<td style="padding:8px 12px;text-align:center;font-weight:600">{row["f1_por_co2"]:.1f}</td>'
        '</tr>'
    )

# Conclusión coste-rendimiento dinámica
ganador_row    = df_sos[df_sos['es_ganador']].iloc[0]
eco_row        = df_sos[df_sos['modelo'] == mejor_eficiencia].iloc[0]
peor_row       = df_sos[df_sos['modelo'] == peor_co2].iloc[0]

texto_conclusion = (
    f'El modelo <strong>{nombre_ganador}</strong> (ganador por F1) emite '
    f'<strong>{ganador_row["co2_g"]:.6f} g</strong> de CO2 por inferencia '
    f'sobre el conjunto de test completo ({len(X_test_prep):,} alumnos). '
    f'<strong>{mejor_eficiencia}</strong> es el más eficiente ambientalmente '
    f'(F1/CO2 = {eco_row["f1_por_co2"]:.1f}). '
    f'<strong>{peor_co2}</strong> es el menos eficiente '
    f'({peor_row["co2_g"]:.6f} g, {peor_row["co2_g"]/max(ganador_row["co2_g"], 1e-9):.1f}× '
    f'más CO2 que el ganador).'
)

contenido = (
    f'<h2 style="color:#2d3748">Fase 6 — M04c · Sostenibilidad Computacional</h2>'
    + bloque_wilcoxon_html(ROOT, nombre_ganador)
    + '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'Análisis del coste computacional y la huella de carbono de los modelos principales '
    f'en inferencia sobre el conjunto de test completo ({len(X_test_prep):,} observaciones). '
    'Las emisiones de CO2 se estiman con codecarbon a partir del consumo energético '
    'y la intensidad de carbono de la red eléctrica local. '
    f'Comparamos el ganador <strong>{nombre_ganador}</strong> con dos puntos representativos '
    'del espectro coste-rendimiento: <strong>Stacking</strong> (ensemble complejo) y '
    '<strong>EBM</strong> (modelo simple e interpretable).'
    '</p>'
    # SOSTENIBILIDAD DESTACADA — bloque verde con conclusión
    f'<div style="margin:20px 0;padding:18px;background:#f0fff4;'
    f'border-left:4px solid {COLOR_EXITO};border-radius:8px;font-size:13px;color:#22543d">'
    f'<h3 style="color:#22543d;font-size:15px;margin-bottom:8px">'
    f'🌱 Conclusión coste-rendimiento</h3>'
    f'<p style="line-height:1.6">{texto_conclusion}</p>'
    f'</div>'
    '<table style="width:100%;border-collapse:collapse;font-size:13px;margin:20px 0">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px;text-align:left">Modelo</th>'
    '<th style="padding:8px 12px;text-align:center">Tiempo total</th>'
    '<th style="padding:8px 12px;text-align:center">ms / obs</th>'
    '<th style="padding:8px 12px;text-align:center">Memoria (MB)</th>'
    '<th style="padding:8px 12px;text-align:center">CO2 (g)</th>'
    '<th style="padding:8px 12px;text-align:center">F1</th>'
    '<th style="padding:8px 12px;text-align:center">F1/CO2 🌱</th>'
    '</tr></thead>'
    f'<tbody>{filas_tabla}</tbody></table>'
    '<p style="color:#718096;font-size:12px;margin-bottom:16px">'
    '🏆 Modelo ganador (mayor F1) · 🌱 Mayor eficiencia ambiental (F1/CO2)'
    '</p>'
    + bloque_imagen(img_b64(ruta_sos),
        f'Comparativa de sostenibilidad ({nombre_ganador} vs Stacking vs EBM)',
        'Tiempo de inferencia por observación, consumo de memoria pico '
        'y emisiones estimadas de CO2 por inferencia completa sobre el conjunto de test.')
    + (bloque_imagen(img_b64(ruta_ratio),
        '🏆 Eficiencia ambiental: F1 por gramo de CO2',
        'Mayor valor = mejor rendimiento por unidad de coste ambiental. '
        'Métrica original que combina rendimiento (F1) con sostenibilidad (CO2). '
        'Útil para defender la elección del modelo ante un comité que valore criterios ESG.'
    ) if ruta_ratio else '')
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    f'border-left:4px solid {COLOR_PRIMARIO};border-radius:6px;font-size:13px;color:#2c5282">'
    '<strong>Contexto:</strong> En un sistema de alerta temprana universitario, '
    'la inferencia se ejecutaría una vez por curso académico sobre todos los alumnos activos. '
    'El coste computacional es, por tanto, marginal en términos absolutos. '
    'La comparativa es relevante si el sistema escala a múltiples universidades o '
    'si se requiere inferencia en tiempo real (consulta alumno a alumno). '
    'Además, el ratio F1/CO2 ofrece un argumento ESG para justificar la elección '
    'del modelo ante criterios de Objetivos de Desarrollo Sostenible.'
    '</div>'
)

ruta_html = DIR_HTML / 'm04c_sostenibilidad.html'
render_pagina(
    'f6_m04c_sostenibilidad.ipynb',
    contenido,
    ruta_html,
    carpeta_notebook='fase6_evaluacion'
)
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: c:\PRUEBAS\AU_UJI_v2_RUTA_B\docs\html\fase6\m04c_sostenibilidad.html
